In [0]:
# COMMAND --- NOTEBOOK

dbutils.widgets.text("catalog", "dev", "Target catalog")
catalog = dbutils.widgets.get("catalog")

# Create the target schema context if it doesn't exist already
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.kruger")

# ---------------------------------------------------------------------------
# 1. Expectations History View (Parsed from your DLT Event Logs)
# ---------------------------------------------------------------------------
# Note: Point storage_location in your DLT pipeline settings to capture these raw event logs
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.kruger.dq_expectations_history AS
SELECT
  origin.pipeline_name,
  origin.flow_name AS table_name,
  timestamp,
  exp.col.name AS rule_name,
  exp.col.dataset,
  exp.col.passed_records,
  exp.col.failed_records,
  ROUND(exp.col.failed_records / NULLIF(exp.col.passed_records + exp.col.failed_records, 0) * 100, 2) AS failure_rate_pct
FROM {catalog}.kruger.ingestion_event_log_raw
CROSS JOIN LATERAL EXPLODE(
  FROM_JSON(details:flow_progress.data_quality.expectations,
    'array<struct<name:string, dataset:string, passed_records:int, failed_records:int>>')
) AS exp
WHERE event_type = 'flow_progress'
""")

print(f"Created {catalog}.kruger.dq_expectations_history")


# ---------------------------------------------------------------------------
# 2. Quarantine Counts View (Simplified to only track your active Bronze quarantine)
# ---------------------------------------------------------------------------
def build_quarantine_select(table_name, target_table):
    full_path = f"{catalog}.kruger.{target_table}"
    if spark.catalog.tableExists(full_path):
        return f"SELECT '{table_name}' AS source, COUNT(*) AS quarantined_count, current_timestamp() AS checked_at FROM {full_path}"
    else:
        # Fallback to zero if the pipeline hasn't run or populated the table yet
        return f"SELECT '{table_name}' AS source, 0 AS quarantined_count, current_timestamp() AS checked_at"

quarantine_queries = [
    build_quarantine_select("bronze_kruger", "bronze_kruger_quarantined"),
]

spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.kruger.dq_quarantine_counts AS
{"\nUNION ALL\n".join(quarantine_queries)}
""")

print(f"Created {catalog}.kruger.dq_quarantine_counts")


# ---------------------------------------------------------------------------
# 3. Referential Integrity View (Tracking orphan metrics between layers)
# ---------------------------------------------------------------------------
# Because you don't have separate dimension tables like products/customers, 
# this view checks for data drops/discrepancies between your Silver table and Gold tables
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.kruger.dq_layer_integrity AS
SELECT 
  'silver_to_gold_h1_h2_gap' AS metric_type, 
  (
    (SELECT COUNT(*) FROM {catalog}.kruger.silver_kruger) - 
    ((SELECT COUNT(*) FROM {catalog}.kruger.gold_kruger_jan_to_jun) + 
     (SELECT COUNT(*) FROM {catalog}.kruger.gold_kruger_jul_to_dec))
  ) AS mismatched_records_count,
  current_timestamp() AS checked_at
""")

print(f"Created {catalog}.kruger.dq_layer_integrity")
